# 🏏 IPL Win Probability - Model Training Pipeline

Welcome to the **Model Training Pipeline** notebook for the **IPL Win Probability Predictor**. In this step, we take our engineered situational features (such as required run rate, match pressure index, and run momentum) and train several classification models to predict the probability of the chasing team winning the match.

### 🎯 Objectives
1. **Data Loading**: Load the feature-engineered training dataset (`training_data.csv`).
2. **Train/Test Split**: Split the dataset into a training set (80%) and a testing set (20%) stratified by the target label to maintain integrity.
3. **Preprocessing Pipeline**: Build a standard `ColumnTransformer` comprising:
   - **`StandardScaler`** for numerical columns (e.g., runs left, balls left, RRR, MPI).
   - **`OneHotEncoder`** for categorical identifiers (batting team, bowling team, venue).
4. **Model Training**: Train and evaluate four distinct classifiers:
   - **Logistic Regression** (baseline, provides clean probabilistic outputs)
   - **Random Forest Classifier** (ensemble model, handles non-linear relationships)
   - **Gradient Boosting Classifier** (sequentially builds estimators to minimize error)
   - **XGBoost Classifier** (extreme gradient boosting, highly optimized and robust)
5. **Model Evaluation & Comparison**: Compile and sort the performance of all models on key metrics (Accuracy, ROC AUC, F1 Score, Precision, Recall).
6. **Save Best Model**: Identify the top-performing model and serialize it (`best_model.pkl` and `model.pkl`) alongside model metadata and performance reports (`metrics.json`).

---  
## ⚙️ 1. Setup & Library Imports
We start by importing core scientific computing libraries and the required scikit-learn and xgboost modules.

In [ ]:
import os
import sys
import json
import time
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Add project path to resolve local configuration modules
sys.path.append(os.path.abspath(".."))
import config

---  
## 📂 2. Data Ingestion
We load the compiled features from the `training_data.csv` dataset. If the dataset has not been generated on disk, we run the project's data orchestrator.

In [ ]:
processed_data_path = os.path.join("..", "data", "processed", "training_data.csv")

if not os.path.exists(processed_data_path):
    print("⚠️ Training dataset not found. Running dataset build flow...")
    import train
    train.verify_project_structure()
    train.generate_synthetic_raw_data()
    train.clean_and_standardize_teams(None, None)
    train.run_feature_engineering()
    processed_data_path = os.path.join("data", "processed", "training_data.csv")

df_raw = pd.read_csv(processed_data_path)
print(f"📊 Loaded training dataset successfully. Shape: {df_raw.shape}")
display(df_raw.head(3))

---  
## 🔀 3. Train/Test Split
We split the dataset into features (`X`) and targets (`y`). We filter out non-predictive columns (such as `match_id` or `city` if redundant) and rename columns to keep the feature names aligned with our prediction schema. We use a **stratified split** to ensure the balance of won versus lost games is identical across both training and testing datasets.

In [ ]:
# Drop non-predictive administrative keys and isolate the target outcome 'result'
X = df_raw.drop(columns=["result"])
y = df_raw["result"]

# Align column naming conventions to match standard API definitions
rename_dict = {
    "target": "total_runs_x",
    "wickets_remaining": "wickets_left"
}
X = X.rename(columns=rename_dict)

# Execute train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"✅ Training feature space shape: {X_train.shape}")
print(f"✅ Testing feature space shape: {X_test.shape}")
print(f"🎯 Target class balance in training set:\n{y_train.value_counts(normalize=True)}")

---  
## 🛠️ 4. Feature Preprocessing Pipeline
To handle categorical and numerical attributes correctly, we configure individual pipelines using scikit-learn's `ColumnTransformer`:
1. **Categorical Pipeline**: Applies `OneHotEncoder` with `handle_unknown='ignore'` to encode batting teams, bowling teams, and venues.
2. **Numerical Pipeline**: Applies `StandardScaler` to bring all engineered numerical features into a standardized scale (mean = 0, variance = 1).

We also dump these individual sub-components so they can be loaded by the web app.

In [ ]:
# Define column mappings
categorical_features = ["batting_team", "bowling_team", "venue"]
numerical_features = [
    "runs_left", "balls_left", "wickets_left", "total_runs_x", 
    "crr", "rrr", "match_pressure_index", "required_boundary_percentage", 
    "run_momentum"
]

feature_columns = categorical_features + numerical_features

# Subset splits to include only designated model inputs
X_train_filtered = X_train[feature_columns].copy()
X_test_filtered = X_test[feature_columns].copy()

# Initialize and fit independent components
scaler = StandardScaler()
scaler.fit(X_train_filtered[numerical_features])

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(X_train_filtered[categorical_features])

# Build the consolidated ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), categorical_features)
    ]
)

# Fit full pipeline to training partition
preprocessor.fit(X_train_filtered)

# Export preprocessing assets
model_dir = os.path.join("..", "models")
os.makedirs(model_dir, exist_ok=True)

joblib.dump(scaler, os.path.join(model_dir, "scaler.pkl"))
joblib.dump(scaler, os.path.join(model_dir, "standard_scaler.pkl"))
joblib.dump(encoder, os.path.join(model_dir, "encoder.pkl"))
joblib.dump(preprocessor, os.path.join(model_dir, "preprocessor.pkl"))

# Save column metadata
feature_metadata = {
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "all_input_features": feature_columns,
    "encoded_feature_names": list(preprocessor.get_feature_names_out())
}
joblib.dump(feature_metadata, os.path.join(model_dir, "feature_columns.pkl"))

print("✨ Preprocessing pipeline built and intermediate artifacts exported!")

---  
## 🤖 5. Model Training & Validation
We write a helper evaluation routine to standardize the fitting, prediction timing, and metric calculation for each candidate model.

In [ ]:
def evaluate_model(model, name):
    """Trains a model and returns a metrics dictionary."""
    # Apply transformations
    X_train_trans = preprocessor.transform(X_train_filtered)
    X_test_trans = preprocessor.transform(X_test_filtered)
    
    # Train time profile
    start_time = time.time()
    model.fit(X_train_trans, y_train)
    training_time = time.time() - start_time
    
    # Predict time profile
    start_time = time.time()
    y_pred = model.predict(X_test_trans)
    prediction_time = time.time() - start_time
    
    # Probability estimates
    y_prob = model.predict_proba(X_test_trans)[:, 1]
    
    # Compute metrics
    metrics = {
        "model_name": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "training_time": training_time,
        "prediction_time": prediction_time,
        "model_object": model
    }
    return metrics

### 1️⃣ Logistic Regression (Baseline Model)
Logistic Regression provides an excellent baseline due to its simple parameters, computational efficiency, and direct probability calibration outputs.

In [ ]:
lr_clf = LogisticRegression(max_iter=1000, random_state=42)
lr_metrics = evaluate_model(lr_clf, "Logistic Regression")
print(f"Logistic Regression Accuracy: {lr_metrics['accuracy']:.4f}")

### 2️⃣ Random Forest Classifier (Bagging Ensemble)
Random Forest handles non-linear patterns well by combining the predictions of multiple decision trees.

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_metrics = evaluate_model(rf_clf, "Random Forest")
print(f"Random Forest Accuracy: {rf_metrics['accuracy']:.4f}")

### 3️⃣ Gradient Boosting Classifier (Boosting Ensemble)
Gradient Boosting trains trees sequentially, with each tree correcting the errors of the previous ones.

In [ ]:
gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_metrics = evaluate_model(gb_clf, "Gradient Boosting")
print(f"Gradient Boosting Accuracy: {gb_metrics['accuracy']:.4f}")

### 4️⃣ XGBoost Classifier (Extreme Gradient Boosting)
XGBoost is a highly optimized gradient boosting implementation known for its speed, scalability, and regularization capabilities to prevent overfitting.

In [ ]:
xgb_clf = XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss")
xgb_metrics = evaluate_model(xgb_clf, "XGBoost")
print(f"XGBoost Accuracy: {xgb_metrics['accuracy']:.4f}")

---  
## 📊 6. Model Evaluation & Comparison
We compile the performance metrics from all candidate classifiers, sort them to identify the top performer, and present a structured model comparison table.

In [ ]:
models_list = [lr_metrics, rf_metrics, gb_metrics, xgb_metrics]

# Sort models based on Accuracy, breaking ties with ROC AUC and F1 Score
sorted_models = sorted(
    models_list,
    key=lambda x: (x["accuracy"], x["roc_auc"], x["f1_score"]),
    reverse=True
)

# Build comparison dataframe
comparison_rows = []
for model in sorted_models:
    comparison_rows.append({
        "Model Name": model["model_name"],
        "Accuracy": model["accuracy"],
        "ROC AUC": model["roc_auc"],
        "F1 Score": model["f1_score"],
        "Precision": model["precision"],
        "Recall": model["recall"],
        "Train Time (s)": model["training_time"]
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

Let's visualize the model accuracies using a Matplotlib bar chart.

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=comparison_df, x="Model Name", y="Accuracy", hue="Model Name", palette="viridis", legend=False)
plt.title("🎯 Classifier Accuracy Comparison")
plt.ylabel("Test Partition Accuracy")
plt.ylim(0, 1.05)
for index, row in comparison_df.iterrows():
    plt.text(index, row["Accuracy"] + 0.02, f"{row['Accuracy']*100:.2f}%", color='black', ha="center")
plt.tight_layout()
plt.show()

---  
## 💾 7. Saving the Best Model
The top-performing model is automatically selected, serialized to disk, and saved to `best_model.pkl` and `model.pkl`. We also compile a comprehensive `metrics.json` file for the application's diagnostic panel.

In [ ]:
best_model_entry = sorted_models[0]
best_model_obj = best_model_entry["model_object"]
best_model_name = best_model_entry["model_name"]

print(f"🏆 Selected Best Model: {best_model_name}")
print(f"   - Accuracy: {best_model_entry['accuracy']:.4f}")
print(f"   - ROC AUC:  {best_model_entry['roc_auc']:.4f}")

# Define save paths
best_model_path = os.path.join(model_dir, "best_model.pkl")
model_export_path = os.path.join(model_dir, "model.pkl")
metrics_json_path = os.path.join(model_dir, "metrics.json")

# Serialize model artifacts
joblib.dump(best_model_obj, best_model_path)
joblib.dump(best_model_obj, model_export_path)

# Calculate feature count and dataset size
X_train_trans = preprocessor.transform(X_train_filtered)
feature_count = int(X_train_trans.shape[1])
dataset_size = int(len(X_train_filtered) + len(X_test_filtered))

# Compile metrics schema
metrics_data = {
    "accuracy": float(best_model_entry["accuracy"]),
    "precision": float(best_model_entry["precision"]),
    "recall": float(best_model_entry["recall"]),
    "f1_score": float(best_model_entry["f1_score"]),
    "roc_auc": float(best_model_entry["roc_auc"]),
    "training_time": float(best_model_entry.get("training_time", 0.0)),
    "prediction_time": float(best_model_entry.get("prediction_time", 0.0)),
    "best_model_name": best_model_name,
    "dataset_size": dataset_size,
    "feature_count": feature_count,
    "all_models_comparison": {}
}

for entry in sorted_models:
    metrics_data["all_models_comparison"][entry["model_name"]] = {
        "accuracy": float(entry["accuracy"]),
        "precision": float(entry["precision"]),
        "recall": float(entry["recall"]),
        "f1_score": float(entry["f1_score"]),
        "roc_auc": float(entry["roc_auc"])
    }

with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=4)

print(f"💾 Best model serialized to {best_model_path}!")
print(f"📊 Run metadata saved to {metrics_json_path}!")